# Parametry a hiperparametry

![Parametry i hiperparametry](../obrazki/2.3_parametry_hiperparametry.png)

Na grafice obok hiperparametrów modelu znalazły się też ustawienia techniczne (np. `test_size`, `random_state`, `cv`). To nie są hiperparametry samego modelu, tylko ustawienia podziału danych i walidacji.

**Plan notatnika:**
1. Hiperparametry – w prostych słowach
2. Parametry modelu
3. Parametry vs hiperparametry – porównanie
4. Przykład w kodzie

## 1. Hiperparametry – w prostych słowach

Hiperparametry to ustawienia modelu wybierane **przed** treningiem. Model sam ich nie uczy z danych.

**Przykłady:**
- drzewo decyzyjne: `max_depth`, `min_samples_split`,
- las losowy: `n_estimators`, `max_depth`,
- sieć neuronowa: `learning_rate`, `batch_size`, liczba epok.

**Po co je ustawiamy?** Kontrolują złożoność modelu (zbyt prosty model → *underfitting*, zbyt złożony → *overfitting*) i szybkość uczenia.

**Jak dobierać hiperparametry?**
1. Zaczynaj od prostych wartości bazowych.
2. Testuj kilka sensownych kombinacji i oceniaj je na walidacji (walidacja krzyżowa).
3. Najlepszą kombinację sprawdź na zbiorze testowym.

**Najczęstsze metody doboru:** **Grid Search** (wszystkie kombinacje z siatki), **Random Search** (losowa część kombinacji), a także Optuna czy Keras Tuner (automatyczne strojenie). Grid Search i walidację krzyżową poznamy dokładniej w rozdziałach 4.4 i 4.5.

### Przykłady hiperparametrów popularnych algorytmów (scikit-learn)

| Algorytm | Przykładowe hiperparametry |
| :--- | :--- |
| Ridge / Lasso | `alpha` (siła regularyzacji) |
| Regresja logistyczna | `C`, `penalty`, `max_iter` |
| k najbliższych sąsiadów (KNN) | `n_neighbors`, `weights`, `metric` |
| Drzewo decyzyjne | `max_depth`, `min_samples_split`, `min_samples_leaf` |
| Las losowy | `n_estimators`, `max_depth`, `max_features` |
| SVM | `C`, `kernel`, `gamma` |
| K-means | `n_clusters` |
| Sieć neuronowa (MLP) | `hidden_layer_sizes`, `learning_rate_init`, `batch_size`, `max_iter` |

## 2. Parametry modelu

Parametry to wartości, które model wyznacza **w trakcie treningu** na podstawie danych.

**Przykłady:**
- regresja liniowa: wagi i wyraz wolny,
- sieć neuronowa: wagi i biasy,
- drzewo decyzyjne: progi i podziały w węzłach.

**Co warto zapamiętać?**
1. Parametry są „wiedzą” modelu o danych.
2. Im bardziej złożony model, tym zwykle więcej parametrów.
3. Zbyt duża swoboda parametrów może prowadzić do overfittingu.
4. Regularyzacja (np. L1/L2) pomaga kontrolować złożoność.

**Skala:** liczba parametrów dużych modeli językowych sięga miliardów.

| Model | Liczba parametrów |
| :--- | :--- |
| GPT-1 (2018) | ok. 117 mln |
| GPT-2 (2019) | ok. 1,5 mld |
| GPT-3 (2020) | ok. 175 mld |
| GPT-4 (2023) | brak oficjalnych danych (nieoficjalne szacunki: rzędu bilionów) |

## 3. Parametry vs hiperparametry – porównanie

| | Parametry | Hiperparametry |
| :--- | :--- | :--- |
| Kto je wyznacza? | model, w trakcie treningu | Ty, **przed** treningiem |
| Przykłady | wagi w regresji, progi podziałów w drzewie | `max_depth`, `n_estimators`, `learning_rate` |
| Intuicja | „co model zapamiętał z danych” | „jak model ma się uczyć” |

## 4. Przykład w kodzie

Poniżej szukamy najlepszych **hiperparametrów** drzewa decyzyjnego (`max_depth`, `min_samples_split`) metodą Grid Search z walidacją krzyżową (`cv=5`: dane treningowe dzielone na 5 części, każda kombinacja oceniana 5 razy). Następnie sprawdzamy **parametry**, których model nauczył się sam: cechę i próg w pierwszym węźle drzewa.

Nie chodzi tu o jak najlepszy wynik, tylko o odróżnienie obu pojęć.

In [1]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Dane
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Hiperparametry, które chcemy sprawdzić
param_grid = {
    'max_depth': [2, 3, 4, None],
    'min_samples_split': [2, 5, 10]
}

# Szukanie najlepszych hiperparametrów
search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5
)
search.fit(X_train, y_train)

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

print('Najlepsze hiperparametry:', search.best_params_)
print(f'Dokładność na zbiorze testowym: {accuracy_score(y_test, y_pred):.3f}')

# Pięć najlepszych kombinacji hiperparametrów
wyniki = pd.DataFrame(search.cv_results_)[['param_max_depth', 'param_min_samples_split', 'mean_test_score']]
print()
print(wyniki.sort_values('mean_test_score', ascending=False).head(5).to_string(index=False))

# Parametry modelu (nauczone z danych): cecha i próg w pierwszym węźle
nazwy_cech = load_iris().feature_names
print()
print(f"Parametry nauczone z danych: pierwszy podział to '{nazwy_cech[best_model.tree_.feature[0]]}' <= {best_model.tree_.threshold[0]:.2f}")

Najlepsze hiperparametry: {'max_depth': 4, 'min_samples_split': 2}
Dokładność na zbiorze testowym: 0.933

param_max_depth  param_min_samples_split  mean_test_score
              4                        2         0.941667
           None                        2         0.941667
              3                        2         0.933333
              3                        5         0.933333
              3                       10         0.933333

Parametry nauczone z danych: pierwszy podział to 'petal length (cm)' <= 2.45


**Jak to czytać:**
- **Hiperparametry** (`max_depth`, `min_samples_split`) ustawiliśmy my – wybraliśmy je z podanej siatki.
- **Parametry** (cecha i próg w węzłach drzewa) wyznaczył model w trakcie uczenia.
- Kilka kombinacji hiperparametrów ma niemal ten sam wynik walidacji. Przy tak małym zbiorze (150 próbek) różnice między nimi są przypadkowe, a wynik na zbiorze testowym (30 próbek) to zaledwie kilka obserwacji. Dlatego w praktyce stosujemy większe zbiory i walidację krzyżową.